<a href="https://colab.research.google.com/github/mdanmek/nida-dads-notes/blob/main/dads5001-data-tools/project/eda/03_construction_eda_2569.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# การสำรวจโครงการจ้างก่อสร้างและการกำหนดกลุ่มศึกษา ปีงบประมาณ 2569

ข้อมูล e-GP ปีงบประมาณ 2569 สะสมถึงวันที่ 30 กรกฎาคม 2569 ไม่ใช่ข้อมูลเต็มปี

Notebook นี้ตอบคำถามตามลำดับดังนี้:

1. โครงการก่อสร้างส่วนใหญ่อยู่ในช่วงวงเงินใด
2. การกระจุกของวงเงินสัมพันธ์กับวิธีจัดซื้อใด
3. กฎหมายอธิบายการกระจุกดังกล่าวได้อย่างไร
4. ควรกำหนดกลุ่มศึกษาสำหรับค้นหา Pattern อย่างไร


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)


In [ ]:
base_dir = Path(
    '/content/drive/MyDrive/learning/dads/dads5001/'
    'project_1_dads5001/dataset/procurement/egp-contract'
)

data_path = base_dir / 'processed' / 'construction_contracts_2569.csv'
project_output_path = base_dir / 'processed' / 'construction_projects_2569.csv'

project_dir = base_dir.parents[2]
figure_dir = project_dir / 'figure'
figure_dir.mkdir(parents=True, exist_ok=True)

print(f'Input file: {data_path}')
print(f'Project output: {project_output_path}')
print(f'Figure directory: {figure_dir}')


In [ ]:
# ดาวน์โหลดฟอนต์สำหรับแสดงภาษาไทยในกราฟ
!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf

fm.fontManager.addfont('thsarabunnew-webfont.ttf')

sns.set_theme(
    style='whitegrid',
    font='TH Sarabun New'
)

plt.rcParams['axes.unicode_minus'] = False

BLUE = '#4C6A85'
AMBER = '#C47A2C'
GRAY = '#A7B1BB'
TEXT = '#263238'
GRID = '#E5E9ED'


## 1. สร้างข้อมูลระดับโครงการ

ไฟล์จาก Notebook 02 อยู่ในระดับ `โครงการ–สัญญา–บริษัท` การสำรวจวงเงินและวิธีจัดซื้อใน Notebook นี้ต้องใช้หนึ่งแถวต่อ `รหัสโครงการ`


In [ ]:
construction_data = pd.read_csv(
    data_path,
    low_memory=False
)

print(f'รายการจาก Notebook 02: {len(construction_data):,}')
print(f'จำนวนคอลัมน์: {construction_data.shape[1]:,}')

display(construction_data.head())


In [ ]:
project_id_column = 'รหัสโครงการ'
budget_column = 'วงเงินงบประมาณ (บาท)'
method_column = 'ชื่อวิธีการจัดซื้อจัดจ้าง'

project_columns = [
    project_id_column,
    'ชื่อโครงการจัดซื้อจัดจ้าง',
    'ชื่อประเภทโครงการ',
    'ชื่อหน่วยงาน',
    'ชื่อหน่วยงานย่อย',
    method_column,
    'ชื่อกลุ่มวิธีการจัดซื้อจัดจ้าง',
    'วันที่ประกาศจัดซื้อจัดจ้าง',
    budget_column,
    'ราคากลาง (บาท)',
    'ราคาที่ตกลงซื้อ / จ้าง ซึ่งรวมทุกสัญญาในโครงการ (บาท)',
    'ปีงบประมาณ',
    'วันที่เกิดรายการ',
    'จังหวัด',
    'เขต/อำเภอ',
    'แขวง/ตำบล',
    'สถานะโครงการ',
    'พิกัดของโครงการ',
    'ละติจูดของโครงการ',
    'ลองจิจูดของโครงการ'
]


In [ ]:
project_data = construction_data[
    project_columns
].drop_duplicates(
    subset=project_id_column,
    keep='first'
).copy()

project_data[budget_column] = pd.to_numeric(
    project_data[budget_column],
    errors='coerce'
)

print(f'รายการระดับสัญญา–บริษัท: {len(construction_data):,}')
print(f'โครงการไม่ซ้ำ: {len(project_data):,}')
print(f'แถวที่รวมเป็นระดับโครงการ: {len(construction_data) - len(project_data):,}')


ข้อมูลสำหรับ EDA เหลือหนึ่งแถวต่อโครงการ ส่วนข้อมูลผู้รับจ้างยังอยู่ในไฟล์จาก Notebook 02 และจะใช้เลขประจำตัวนิติบุคคลเป็นคีย์ใน Notebook 04


## 2. โครงการก่อสร้างส่วนใหญ่อยู่ช่วงใด

เริ่มจากดูค่ากลาง ช่วงวงเงิน และวงเงินที่พบซ้ำบ่อย โดยยังไม่ใช้กฎหมายเป็นจุดตั้งต้น


In [ ]:
budget_summary = project_data[budget_column].describe(
    percentiles=[0.25, 0.50, 0.75, 0.95, 0.99]
)

display(budget_summary.to_frame(name='วงเงินงบประมาณ'))

print(f'Mean: {project_data[budget_column].mean():,.0f} บาท')
print(f'Median: {project_data[budget_column].median():,.0f} บาท')
print(f'99th percentile: {project_data[budget_column].quantile(0.99):,.0f} บาท')


In [ ]:
common_budget_values = (
    project_data[budget_column]
    .value_counts()
    .head(15)
    .rename_axis('วงเงินงบประมาณ')
    .reset_index(name='จำนวนโครงการ')
)

common_budget_values['สัดส่วนโครงการ (%)'] = (
    common_budget_values['จำนวนโครงการ']
    / len(project_data)
    * 100
)

display(common_budget_values)


In [ ]:
budget_bins = [
    0,
    100_000,
    200_000,
    300_000,
    400_000,
    500_000,
    1_000_000,
    5_000_000,
    10_000_000,
    50_000_000,
    np.inf
]

budget_labels = [
    'ไม่เกิน 100,000',
    '100,001–200,000',
    '200,001–300,000',
    '300,001–400,000',
    '400,001–500,000',
    '500,001–1 ล้าน',
    'มากกว่า 1–5 ล้าน',
    'มากกว่า 5–10 ล้าน',
    'มากกว่า 10–50 ล้าน',
    'มากกว่า 50 ล้าน'
]

project_data['budget_band'] = pd.cut(
    project_data[budget_column],
    bins=budget_bins,
    labels=budget_labels,
    include_lowest=True
)


In [ ]:
budget_band_summary = (
    project_data
    .groupby('budget_band', observed=False)
    .agg(
        project_count=(project_id_column, 'size'),
        total_budget=(budget_column, 'sum')
    )
    .reset_index()
)

budget_band_summary['project_pct'] = (
    budget_band_summary['project_count']
    / len(project_data)
    * 100
)

budget_band_summary['budget_pct'] = (
    budget_band_summary['total_budget']
    / budget_band_summary['total_budget'].sum()
    * 100
)

display(budget_band_summary)


In [ ]:
budget_plot = budget_band_summary.set_index('budget_band')[
    ['project_pct', 'budget_pct']
]

budget_plot.columns = [
    'สัดส่วนจำนวนโครงการ',
    'สัดส่วนวงเงินรวม'
]

ax = budget_plot.plot.barh(
    figsize=(12, 7),
    color=[BLUE, AMBER],
    width=0.75
)

for bars in ax.containers:
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=11)

ax.invert_yaxis()
ax.set_title(
    'โครงการขนาดเล็กครองจำนวน แต่โครงการขนาดใหญ่ครองวงเงิน',
    loc='left',
    fontsize=19,
    fontweight='bold',
    color=TEXT
)
ax.set_xlabel('สัดส่วน (%)')
ax.set_ylabel('ช่วงวงเงินงบประมาณ')
ax.legend(title='')
ax.grid(axis='x', color=GRID)
ax.grid(axis='y', visible=False)
sns.despine(left=True, bottom=True)

fig = ax.get_figure()
fig.tight_layout()


In [ ]:
png_path = figure_dir / 'fig03_01_budget_band_count_vs_value.png'
svg_path = figure_dir / 'fig03_01_budget_band_count_vs_value.svg'

fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')


### สิ่งที่พบ

วงเงินมัธยฐานอยู่ที่ 395,000 บาท ขณะที่ค่าเฉลี่ยประมาณ 2.66 ล้านบาท แสดงว่าการกระจายเบ้ขวาจากโครงการขนาดใหญ่จำนวนน้อย

โครงการไม่เกิน 500,000 บาทมี 136,623 โครงการ หรือ 76.34% ของโครงการทั้งหมด แต่ครองวงเงินรวมประมาณ 8.23% โดยช่วง 400,001–500,000 บาทมีจำนวนโครงการมากที่สุด และวงเงิน 500,000 บาทเป็นค่าที่พบซ้ำบ่อยที่สุด

คำถามถัดไปคือ การกระจุกของโครงการขนาดเล็กสัมพันธ์กับวิธีจัดซื้อใด


## 3. การกระจุกสัมพันธ์กับวิธีจัดซื้อใด

เปรียบเทียบแต่ละวิธีทั้งด้านจำนวนโครงการและวงเงินรวม เพื่อแยก “วิธีที่ใช้บ่อย” ออกจาก “วิธีที่ครองมูลค่า”


In [ ]:
method_summary = (
    project_data
    .groupby(method_column)
    .agg(
        project_count=(project_id_column, 'size'),
        total_budget=(budget_column, 'sum'),
        median_budget=(budget_column, 'median')
    )
    .reset_index()
)

method_summary['project_pct'] = (
    method_summary['project_count']
    / len(project_data)
    * 100
)

method_summary['budget_pct'] = (
    method_summary['total_budget']
    / method_summary['total_budget'].sum()
    * 100
)

method_summary = method_summary.sort_values(
    'project_count',
    ascending=False
)

display(method_summary)


In [ ]:
method_names = {
    'เฉพาะเจาะจง': 'เฉพาะเจาะจง',
    'ประกวดราคาอิเล็กทรอนิกส์ (e-bidding)': 'e-bidding',
    'คัดเลือก': 'คัดเลือก',
    'ตกลงราคา': 'ตกลงราคา'
}

method_plot = method_summary.copy()
method_plot['method_label'] = (
    method_plot[method_column]
    .map(method_names)
    .fillna(method_plot[method_column])
)

method_plot = method_plot.set_index('method_label')[
    ['project_pct', 'budget_pct']
]

method_plot.columns = [
    'สัดส่วนจำนวนโครงการ',
    'สัดส่วนวงเงินรวม'
]


In [ ]:
ax = method_plot.plot.barh(
    figsize=(11, 5.5),
    color=[BLUE, AMBER],
    width=0.72
)

for bars in ax.containers:
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=11)

ax.invert_yaxis()
ax.set_title(
    'วิธีเฉพาะเจาะจงครองจำนวน ขณะที่ e-bidding ครองวงเงิน',
    loc='left',
    fontsize=19,
    fontweight='bold',
    color=TEXT
)
ax.set_xlabel('สัดส่วน (%)')
ax.set_ylabel('')
ax.legend(title='')
ax.grid(axis='x', color=GRID)
ax.grid(axis='y', visible=False)
sns.despine(left=True, bottom=True)

fig = ax.get_figure()
fig.tight_layout()


In [ ]:
png_path = figure_dir / 'fig03_02_method_count_vs_value.png'
svg_path = figure_dir / 'fig03_02_method_count_vs_value.svg'

fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')


### สิ่งที่พบ

วิธีเฉพาะเจาะจงคิดเป็น 76.35% ของจำนวนโครงการ แต่ใช้วงเงินประมาณ 9.33% ขณะที่ e-bidding มี 21.22% ของจำนวนโครงการและครองวงเงินประมาณ 83.34%

ผลนี้เชื่อมการกระจุกของโครงการขนาดเล็กกับวิธีเฉพาะเจาะจง จึงต้องตรวจบริบทกฎหมายก่อนตีความว่าเป็นความผิดปกติ


## 4. กฎหมายอธิบายการกระจุกอย่างไร

เมื่อนำผลจากข้อมูลไปเทียบกับกฎหมาย พบว่า:

- พระราชบัญญัติการจัดซื้อจัดจ้างฯ พ.ศ. 2560 มาตรา 55 กำหนดวิธีจัดซื้อจัดจ้างพัสดุ 3 วิธี ได้แก่ วิธีประกาศเชิญชวนทั่วไป วิธีคัดเลือก และวิธีเฉพาะเจาะจง
- มาตรา 56 วรรคหนึ่ง (2)(ข) เปิดให้ใช้วิธีเฉพาะเจาะจงกับพัสดุทั่วไป เมื่อวงเงินต่อครั้งไม่เกินวงเงินที่กำหนดในกฎกระทรวง
- กฎกระทรวงกำหนดวงเงินดังกล่าวไว้ไม่เกิน 500,000 บาท

แหล่งอ้างอิง: [พระราชบัญญัติการจัดซื้อจัดจ้างฯ ในราชกิจจานุเบกษา](https://www.ratchakitcha.soc.go.th/) และ [กฎหมาย/ระเบียบด้านการจัดซื้อจัดจ้างของกรมบัญชีกลาง](https://www.cgd.go.th/)

ดังนั้น การกระจุกของวิธีเฉพาะเจาะจงใต้ 500,000 บาทเป็นรูปแบบเชิงโครงสร้างที่กฎหมายอธิบายได้ ไม่ใช่ anomaly ด้วยตัวเอง


## 5. กำหนดกลุ่มศึกษาหลัก

เพื่อค้นหา Pattern ที่กฎหมายยังอธิบายไม่ได้ จะเปรียบเทียบโครงการภายใต้บริบทเดียวกัน คือวิธีเฉพาะเจาะจงและวงเงินไม่เกิน 500,000 บาท


In [ ]:
under_500k_mask = project_data[budget_column] <= 500_000

study_population_mask = (
    under_500k_mask
    & project_data[method_column].eq('เฉพาะเจาะจง')
)

project_data['is_study_population'] = study_population_mask
study_project_data = project_data[study_population_mask].copy()

study_scope = pd.DataFrame({
    'ขั้นการเลือกข้อมูล': [
        'โครงการก่อสร้างทั้งหมด',
        'วงเงินไม่เกิน 500,000 บาท',
        'เฉพาะเจาะจงและไม่เกิน 500,000 บาท'
    ],
    'project_count': [
        len(project_data),
        under_500k_mask.sum(),
        len(study_project_data)
    ]
})

study_scope['project_pct'] = (
    study_scope['project_count']
    / len(project_data)
    * 100
)

display(study_scope)


In [ ]:
scope_colors = [GRAY, BLUE, AMBER]

fig, ax = plt.subplots(figsize=(11, 5.5))

bars = ax.barh(
    study_scope['ขั้นการเลือกข้อมูล'],
    study_scope['project_count'],
    color=scope_colors,
    height=0.58
)

labels = [
    f'{count:,.0f} โครงการ ({pct:.1f}%)'
    for count, pct in zip(
        study_scope['project_count'],
        study_scope['project_pct']
    )
]

ax.bar_label(bars, labels=labels, padding=5, fontsize=12)
ax.invert_yaxis()
ax.set_title(
    'จากโครงการก่อสร้างทั้งหมดสู่กลุ่มศึกษาภายใต้บริบทเดียวกัน',
    loc='left',
    fontsize=19,
    fontweight='bold',
    color=TEXT
)
ax.set_xlabel('จำนวนโครงการ')
ax.set_ylabel('')
ax.set_xlim(0, study_scope['project_count'].max() * 1.25)
ax.grid(axis='x', color=GRID)
ax.grid(axis='y', visible=False)
sns.despine(left=True, bottom=True)

fig.tight_layout()


In [ ]:
png_path = figure_dir / 'fig03_03_study_scope.png'
svg_path = figure_dir / 'fig03_03_study_scope.svg'

fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')


### ผลการกำหนดขอบเขต

กลุ่มศึกษาลดจากโครงการก่อสร้างทั้งหมด 178,978 โครงการ เหลือ 136,623 โครงการที่มีวงเงินไม่เกิน 500,000 บาท และเหลือ 136,070 โครงการเมื่อจำกัดเฉพาะวิธีเฉพาะเจาะจง

การกำหนดขอบเขตนี้ไม่ได้หมายความว่า 136,070 โครงการผิดปกติ แต่ทำให้การค้นหา Pattern เปรียบเทียบโครงการที่อยู่ภายใต้เงื่อนไขกฎหมายเดียวกัน


## 6. การกระจายรอบเส้น 500,000 บาท

ดูรายละเอียดช่วง 400,000–550,000 บาท เพื่อระบุตำแหน่งการกระจุก และเตรียมทดสอบช่วงใกล้เพดานใน Notebook 04


In [ ]:
threshold_data = project_data[
    project_data[budget_column].between(
        400_000,
        550_000,
        inclusive='both'
    )
].copy()

threshold_data['method_group'] = 'วิธีอื่น'

specific_mask = threshold_data[method_column].eq('เฉพาะเจาะจง')
threshold_data.loc[specific_mask, 'method_group'] = 'เฉพาะเจาะจง'

print(f'โครงการช่วง 400,000–550,000 บาท: {len(threshold_data):,}')


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

sns.histplot(
    data=threshold_data,
    x=budget_column,
    hue='method_group',
    binwidth=10_000,
    multiple='stack',
    palette={
        'เฉพาะเจาะจง': BLUE,
        'วิธีอื่น': GRAY
    },
    ax=ax
)

ax.axvspan(
    490_000,
    500_000,
    color=AMBER,
    alpha=0.18
)

ax.axvline(
    500_000,
    color=AMBER,
    linestyle='--',
    linewidth=2
)

ax.text(
    501_500,
    ax.get_ylim()[1] * 0.92,
    '500,000 บาท',
    color=AMBER,
    fontsize=12
)

ax.set_title(
    'จำนวนโครงการกระจุกบริเวณก่อนถึง 500,000 บาท',
    loc='left',
    fontsize=19,
    fontweight='bold',
    color=TEXT
)
ax.set_xlabel('วงเงินงบประมาณ (บาท)')
ax.set_ylabel('จำนวนโครงการ')
ax.ticklabel_format(style='plain', axis='x')
ax.grid(axis='y', color=GRID)
ax.grid(axis='x', visible=False)
sns.despine()

fig.tight_layout()


In [ ]:
png_path = figure_dir / 'fig03_04_distribution_around_500k.png'
svg_path = figure_dir / 'fig03_04_distribution_around_500k.svg'

fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')


In [ ]:
near_threshold_windows = [
    (450_000, 500_000, '450,000–500,000'),
    (480_000, 500_000, '480,000–500,000'),
    (490_000, 500_000, '490,000–500,000'),
    (495_000, 500_000, '495,000–500,000')
]

window_results = []

for lower, upper, label in near_threshold_windows:
    project_count = study_project_data[budget_column].between(
        lower,
        upper,
        inclusive='both'
    ).sum()

    window_results.append({
        'ช่วงวงเงิน': label,
        'จำนวนโครงการ': project_count,
        'สัดส่วนกลุ่มศึกษา (%)': (
            project_count / len(study_project_data) * 100
        )
    })

near_threshold_summary = pd.DataFrame(window_results)

display(near_threshold_summary)


### สิ่งที่ส่งต่อไป Notebook 04

ช่วง 490,000–500,000 บาทมี 26,154 โครงการ และพบค่าที่เกิดบ่อยหลายค่าบริเวณก่อนถึงเพดาน เช่น 490,000, 495,000, 498,000, 499,000 และ 500,000 บาท

500,000 บาทมีที่มาจากกฎหมาย แต่ 490,000 บาทเป็นขอบล่างที่กำหนดเพื่อการศึกษา จึงต้องทดสอบหลายช่วงร่วมกับเงื่อนไขการเกิดซ้ำใน Notebook 04


In [ ]:
project_data.to_csv(
    project_output_path,
    index=False,
    encoding='utf-8-sig'
)

print(f'Project rows saved: {len(project_data):,}')
print(f'Study population: {len(study_project_data):,}')
print(f'Saved to: {project_output_path}')


## 7. สรุปและคำถามส่งต่อ

สิ่งที่กฎหมายอธิบายได้คือ การใช้วิธีเฉพาะเจาะจงและการกระจุกใต้เพดาน 500,000 บาท ส่วนสิ่งที่กฎหมายยังไม่อธิบายคือ:

- เหตุใดบางคู่หน่วยงาน–ผู้รับจ้างจึงมีหลายโครงการใกล้เพดานในวันเดียวกัน
- เหตุใดบางหน่วยงานย่อยจึงพึ่งพาผู้รับจ้างรายเดียวในสัดส่วนสูง

Notebook 04 จะเริ่มจากกลุ่มศึกษา 136,070 โครงการ และตรวจสอง Pattern นี้ โดยใช้ `เลขประจำตัวนิติบุคคล 13 หลัก` เป็นคีย์ผู้รับจ้าง ส่วนชื่อผู้ชนะใช้เพื่อแสดงผลเท่านั้น
